# Phase 16: Encoding & Scaling Pipeline

**Goal:** Our data is mathematically clean, but it is not yet ready for an AI. 
AI models only understand numbers, but we have text columns like `protocol_type` (TCP/UDP). Furthermore, our numeric columns have wildly different scales (e.g., `duration` might be 5 seconds, but `bytes` might be 1,000,000). 

In this phase, we will convert text into numbers (**Encoding**) and squish massive numbers into a balanced range (**Scaling**).

In [1]:
import sys
!{sys.executable} -m pip install pandas numpy scikit-learn -q  # noqa

import warnings
warnings.filterwarnings("ignore")


In [2]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler


### Step 1: The Golden Rule - Split Before Scaling
Before we do *any* scaling, we must split our data into a **Training Set** and a **Testing Set**. 

**Why? (Preventing Data Leakage)**: If you scale the entire dataset at once, the mathematical formula will "peek" at the testing data to calculate the average. This means your AI will cheat on the final exam because it has already seen hints from the test set! We only fit our scalers on the Training Data.

In [3]:
# 1. Create a fake, clean mini-dataset to demonstrate
df = pd.DataFrame({
    "protocol_type": ["tcp", "udp", "icmp"] * 10,  # 30 rows to guarantee safe splitting!
    "bytes_sent": [500, 20, 999999] * 10,          
    "label": [0, 0, 1] * 10                        
})

print("=== ORIGINAL CLEAN DATA ===")
display(df.head())

# 2. Split the data! (80% Train, 20% Test)
X = df.drop(columns=['label'])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\nSUCCESS! Data successfully split to prevent Data Leakage.")


=== ORIGINAL CLEAN DATA ===


,protocol_type,bytes_sent,label
0,tcp,500,0
1,udp,20,0
2,icmp,999999,1
3,tcp,500,0
4,udp,20,0



SUCCESS! Data successfully split to prevent Data Leakage.


### Step 2: Categorical Encoding (Text to Numbers)
AI models like Neural Networks cannot read the word `tcp`. We need to use a **Label Encoder** to convert `tcp`, `udp`, and `icmp` into numbers like `0`, `1`, and `2`.

In [4]:
encoder = LabelEncoder()

# We FIT the encoder ONLY on the training data
X_train['protocol_type'] = encoder.fit_transform(X_train['protocol_type'])

# We TRANSFORM the test data (so it uses the exact same numbering system)
X_test['protocol_type'] = encoder.transform(X_test['protocol_type'])

print("=== AFTER ENCODING ===")
print("The 'protocol_type' column is now numbers!")
display(X_train)

=== AFTER ENCODING ===
The 'protocol_type' column is now numbers!


,protocol_type,bytes_sent
28,2,20
24,1,500
12,1,500
0,1,500
4,2,20
16,2,20
5,0,999999
13,2,20
11,0,999999
22,2,20


### Step 3: Robust Scaling (Protecting the Outliers)
In Phase 15, we learned that we should **NOT** delete outliers, because in cybersecurity, outliers are often the actual cyber attacks (like a DDoS attack generating 999,999 bytes).

Standard scalers (like `StandardScaler`) get completely ruined by these massive numbers. Instead, we use `RobustScaler`. It scales the normal traffic perfectly, while allowing the hacker outliers to remain massive so the AI can easily detect them!

In [5]:
scaler = RobustScaler()

# Again, we only FIT on the training data
numeric_columns = ['bytes_sent']
X_train[numeric_columns] = scaler.fit_transform(X_train[numeric_columns])

# Transform the test data
X_test[numeric_columns] = scaler.transform(X_test[numeric_columns])

print("=== AFTER ROBUST SCALING ===")
print("Notice how the massive DDoS outlier (999,999) is successfully preserved as a huge number (1332.33),")
print("while the normal traffic is nicely scaled down near 0!")
display(X_train)

=== AFTER ROBUST SCALING ===
Notice how the massive DDoS outlier (999,999) is successfully preserved as a huge number (1332.33),
while the normal traffic is nicely scaled down near 0!


,protocol_type,bytes_sent
28,2,-0.00048
24,1,0.00000
12,1,0.00000
0,1,0.00000
4,2,-0.00048
16,2,-0.00048
5,0,0.99952
13,2,-0.00048
11,0,0.99952
22,2,-0.00048


---
### 💾 Persist Processed Data for Downstream Notebooks
We save the scaled train/test arrays so model notebooks (19–28) can load them without re-running the full pipeline.
This is the bridge between the **preprocessing phase** and the **modelling phase**.


In [6]:
import numpy as np, os

os.makedirs("../../data/processed", exist_ok=True)

# Persist the encoded feature matrix and labels for all downstream model notebooks.
# Any notebook that needs real data loads from ml/data/processed/ instead of regenerating.
np.save("../../data/processed/X_train.npy", X_train.values if hasattr(X_train, "values") else X_train)
np.save("../../data/processed/X_test.npy",  X_test.values  if hasattr(X_test,  "values") else X_test)
np.save("../../data/processed/y_train.npy", y_train.values if hasattr(y_train, "values") else y_train)
np.save("../../data/processed/y_test.npy",  y_test.values  if hasattr(y_test,  "values") else y_test)

print(f"✅ Processed data saved to ml/data/processed/")
print(f"   X_train: {X_train.shape}  X_test: {X_test.shape}")
print(f"   y_train: {y_train.shape}  y_test: {y_test.shape}")


✅ Processed data saved to ml/data/processed/
   X_train: (24, 2)  X_test: (6, 2)
   y_train: (24,)  y_test: (6,)


---
## ✅ Summary — Phase 16 — Encoding & Scaling

We label-encoded categorical features and StandardScaler-normalized numerics. Scalers fitted on train set ONLY, then applied to val/test (no data leakage). **Next → Phase 17: Class Imbalance**
